# Season outputs loader
Helper cells to flexibly load season run outputs stored under `data/season_outputs/<run_id>`.
- Lists available runs
- Loads trip log, day summary, season person snapshots, SP day summary
- Discovers all `day_*_model_ts.parquet` files into a dict keyed by day index

Update `RUN_ID` below to point at the run you want to analyze.

In [ ]:
from pathlib import Path
import re
import pandas as pd

BASE_DIR = Path('data/season_outputs')
RUN_ID = None  # e.g., 'my_season_id'

def list_runs(base_dir: Path = BASE_DIR):
    if not base_dir.exists():
        return []
    return sorted([p.name for p in base_dir.iterdir() if p.is_dir()])

def _read_parquet_optional(path: Path):
    if not path.exists():
        return None
    return pd.read_parquet(path)

def load_model_ts(run_dir: Path):
    model_ts = {}
    if not run_dir.exists():
        return model_ts
    pattern = re.compile(r'^day_(\d+)_model_ts\.parquet$')
    for p in sorted(run_dir.glob('day_*_model_ts.parquet')):
        m = pattern.match(p.name)
        if not m:
            continue
        day_idx = int(m.group(1))
        model_ts[day_idx] = pd.read_parquet(p)
    return model_ts

def load_run(run_id: str, base_dir: Path = BASE_DIR):
    run_dir = base_dir / run_id
    data = {
        'run_dir': run_dir,
        'trip_log': _read_parquet_optional(run_dir / 'trip_log.parquet'),
        'day_summary': _read_parquet_optional(run_dir / 'day_summary.parquet'),
        'season_person_log': _read_parquet_optional(run_dir / 'season_person_log.parquet'),
        'sp_day_summary': _read_parquet_optional(run_dir / 'sp_day_summary.parquet'),
        'model_ts': load_model_ts(run_dir),
    }
    return data

available_runs = list_runs()
available_runs


## Load a run
Set `RUN_ID` to one of the `available_runs` above.

In [ ]:
if RUN_ID is None:
    raise ValueError('Set RUN_ID to one of available_runs before loading.')

run_data = load_run(RUN_ID)
run_data_keys = {k: (list(v.keys()) if k == 'model_ts' else (None if v is None else getattr(v, 'shape', None))) for k, v in run_data.items()}
run_data_keys


## Quick peeks
Uncomment and run the snippets you need once a run is loaded.

In [ ]:
# trip_log = run_data['trip_log']
# day_summary = run_data['day_summary']
# season_person_log = run_data['season_person_log']
# sp_day_summary = run_data['sp_day_summary']
# model_ts = run_data['model_ts']

# display(day_summary.head()) if day_summary is not None else None
# display(trip_log.head()) if trip_log is not None else None
# display(sp_day_summary.head()) if sp_day_summary is not None else None
# list(model_ts.keys())
